<a href="https://colab.research.google.com/github/uniesecruz/Analise_Income_Census/blob/main/TCC_ESALQ.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Instalação das bibliotecas

In [ ]:
# Instalar pyspark
!pip install pyspark

# Leitura do arquivos CSV com spark

In [ ]:
from pyspark.sql import SparkSession

# Inicializar a SparkSession
spark = SparkSession.builder.appName("CSV_Reader").getOrCreate()

print("SparkSession inicializada.")

SparkSession inicializada.


In [ ]:
# Carregar o arquivo CSV usando Spark
spark_trans_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Large_Trans.csv', header=True, inferSchema=True)

print("Schema do Spark DataFrame de Transações:")
spark_trans_df.printSchema()

print("Primeiras 5 linhas do Spark DataFrame de Transações:")
spark_trans_df.show(5)

Schema do Spark DataFrame de Transações:
root
 |-- Timestamp: string (nullable = true)
 |-- From Bank: integer (nullable = true)
 |-- Account2: string (nullable = true)
 |-- To Bank: integer (nullable = true)
 |-- Account4: string (nullable = true)
 |-- Amount Received: double (nullable = true)
 |-- Receiving Currency: string (nullable = true)
 |-- Amount Paid: double (nullable = true)
 |-- Payment Currency: string (nullable = true)
 |-- Payment Format: string (nullable = true)
 |-- Is Laundering: integer (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Transações:
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+----------------+--------------+-------------+
|       Timestamp|From Bank| Account2|To Bank| Account4|Amount Received|Receiving Currency|Amount Paid|Payment Currency|Payment Format|Is Laundering|
+----------------+---------+---------+-------+---------+---------------+------------------+-----------+--------------

In [ ]:
# Carregar o arquivo CSV de contas usando Spark
spark_accounts_df = spark.read.csv('/content/drive/MyDrive/TCC/data/external/HI-Large_accounts.csv', header=True, inferSchema=True)

print("Schema do Spark DataFrame de Contas:")
spark_accounts_df.printSchema()

print("Primeiras 5 linhas do Spark DataFrame de Contas:")
spark_accounts_df.show(5)

Schema do Spark DataFrame de Contas:
root
 |-- Bank Name: string (nullable = true)
 |-- Bank ID: integer (nullable = true)
 |-- Account Number: string (nullable = true)
 |-- Entity ID: string (nullable = true)
 |-- Entity Name: string (nullable = true)

Primeiras 5 linhas do Spark DataFrame de Contas:
+--------------------+-------+--------------+---------+--------------------+
|           Bank Name|Bank ID|Account Number|Entity ID|         Entity Name|
+--------------------+-------+--------------+---------+--------------------+
| Portugal Bank #4507| 331579|     80B779D80|80062E240|Sole Proprietorsh...|
|     Canada Bank #27|    210|     809D86900|800C998A0|  Corporation #33520|
|         UK Bank #33|  21884|     80812BE00|800C47F50|  Partnership #35397|
|  Germany Bank #4815|  32742|     81047F300|80096F0B0|  Corporation #48813|
|National Bank of ...| 127390|     80BD8CF00|800FB8760|    Corporation #889|
+--------------------+-------+--------------+---------+--------------------+
only

Salvando os arquivos em parquet

In [ ]:
spark_trans_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Large_Trans', mode='overwrite')
print("spark_trans_df salvo em parquet.")

spark_accounts_df.write.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Large_accounts', mode='overwrite')
print("spark_accounts_df salvo em parquet.")

spark_trans_df salvo em parquet.
spark_accounts_df salvo em parquet.


# Lendo o arquivo parquet (Começar aqui)

In [ ]:
from pyspark.sql import SparkSession
import os

# 1. Definir a memória máxima para o driver (essencial no Colab)
# Deixamos uma margem de segurança para o Sistema Operacional (~4-5GB)
memory_limit = "46g"

spark = SparkSession.builder \
    .appName("Max_Performance_Spark") \
    .config("spark.driver.memory", memory_limit) \
    .config("spark.executor.memory", memory_limit) \
    .config("spark.driver.maxResultSize", "10g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.memory.fraction", "0.8") \
    .config("spark.memory.storageFraction", "0.3") \
    .config("spark.ui.port", "4050") \
    .getOrCreate()

print(f"SparkSession inicializada com foco em High-RAM ({memory_limit}).")

SparkSession inicializada com foco em High-RAM (46g).


In [ ]:
spark_trans_df= spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Large_Trans')

In [ ]:
spark_accounts_df = spark.read.parquet('/content/drive/MyDrive/TCC/data/parquet/HI-Large_accounts')


# Profiling Trans

In [ ]:
print("### Profiling do spark_trans_df ###")

print("\nSchema do DataFrame:")
spark_trans_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_trans_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_trans_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_trans_df.summary().show()

print(f"\nNúmero total de linhas: {spark_trans_df.count()}")

print("\nNomes das Colunas:")
print(spark_trans_df.columns)

# Profiling Accounts

In [ ]:
print("### Profiling do spark_accounts_df ###")

print("\nSchema do DataFrame:")
spark_accounts_df.printSchema()

print("\nTipos de Dados das Colunas:")
for col, dtype in spark_accounts_df.dtypes:
    print(f"{col}: {dtype}")

print("\nEstatísticas Descritivas (describe()):")
spark_accounts_df.describe().show()

print("\nEstatísticas Sumárias (summary()):")
spark_accounts_df.summary().show()

print(f"\nNúmero total de linhas: {spark_accounts_df.count()}")

print("\nNomes das Colunas:")
print(spark_accounts_df.columns)

# Join dos datasets

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

# 1. Configuração de Schema Manual (Recomendado pelo artigo para evitar overhead)
trans_schema = StructType([
    StructField("Timestamp", StringType(), True),
    StructField("From Bank", StringType(), True),
    StructField("from_account", StringType(), True), # Renomeado na leitura
    StructField("To Bank", StringType(), True),
    StructField("to_account", StringType(), True),   # Renomeado na leitura
    StructField("Amount Received", DoubleType(), True),
    StructField("Receiving Currency", StringType(), True),
    StructField("Amount Paid", DoubleType(), True),
    StructField("Payment Currency", StringType(), True),
    StructField("Payment Format", StringType(), True),
    StructField("Is Laundering", IntegerType(), True)
])

# 2. Carregamento dos Dados
# Dica: No HI-Medium, use 'from_account' e 'to_account' para evitar nomes duplicados
# spark_trans_df = spark.read.csv("HI-Medium_Trans.csv", header=True, schema=trans_schema)
# spark_accounts_df = spark.read.csv("HI-Medium_accounts.csv", header=True, inferSchema=True)

# 3. Double Join com Broadcasting (Técnica de Big Data para o TCC)
# O artigo destaca que o contexto da entidade é vital.
# Vamos trazer os dados da conta de ORIGEM e DESTINO.

# Preparando tabela de contas para o Join
accounts_info = spark_accounts_df.select(
    F.col("Account Number").alias("acc_id"),
    F.col("Entity Name").alias("entity"),
    F.col("Bank Name").alias("bank_name")
)

# Join para o Remetente (Sender)
df_enriched = spark_trans_df.join(
    F.broadcast(accounts_info), # Otimização crucial para HI-Medium
    spark_trans_df.from_account == accounts_info.acc_id,
    "left"
).select(spark_trans_df["*"],
         F.col("entity").alias("sender_entity"),
         F.col("bank_name").alias("sender_bank_name")
).drop("acc_id")

# Join para o Destinatário (Receiver)
df_final = df_enriched.join(
    F.broadcast(accounts_info),
    df_enriched.to_account == accounts_info.acc_id,
    "left"
).select(df_enriched["*"],
         F.col("entity").alias("receiver_entity"),
         F.col("bank_name").alias("receiver_bank_name")
).drop("acc_id")

# 4. Verificação de Integridade
print(f"Total de Transações: {df_final.count()}")
df_final.show(5)